# Statistics for WLASL: Pre-processed

In [ ]:
import json
from pathlib import Path
from typing import cast, List, TypedDict, Dict, Tuple
import matplotlib.pyplot as plt
import numpy as np
#locals
from configs import WLASL_ROOT, SPLIT_DIR, CLASSES_PATH
from video_dataset import get_wlasl_info
import stats
import preprocess as preproc
from statistics import median, mean, median_high, median_low
import pandas as pd      # not imported previously, so do it here
from IPython.display import display



## From the WLASL GitHUB page

Data Description
-----------------

* `gloss`: *str*, data file is structured/categorised based on sign gloss, or namely, labels.

* `bbox`: *[int]*, bounding box detected using YOLOv3 of (xmin, ymin, xmax, ymax) convention. Following OpenCV convention, (0, 0) is the up-left corner.

* `fps`: *int*, frame rate (=25) used to decode the video as in the paper.

* `frame_start`: *int*, the starting frame of the gloss in the video (decoding
with FPS=25), *indexed from 1*.

* `frame_end`: *int*, the ending frame of the gloss in the video (decoding with FPS=25). -1 indicates the gloss ends at the last frame of the video.

* `instance_id`: *int*, id of the instance in the same class/gloss.

* `signer_id`: *int*, id of the signer.

* `source`: *str*, a string identifier for the source site.

* `split`: *str*, indicates sample belongs to which subset.

* `url`: *str*, used for video downloading.

* `variation_id`: *int*, id for dialect (indexed from 0).

* `video_id`: *str*, a unique video identifier.


### Additional info:

* The videos come pre-cut from the original youtube videos, therefore, the video_id is essentially a unique identifier for each instance

* There are some issues with the labelling, especially where certain frame start and ends are way too high. However, some of these labels could be corrected by hand to falvage the data. Originally, i just set the start frame to 0 and the end frame to the last frame, however in the modified preprocessing script, i have added a flag to strictly remove these labels for the time being

* All precut video clips have a width and height of 256 pixels. 

* i modified boundign boxes with yolov8


### Naming convention:
For the naming of different functions, 'set' and 'split' can somtimes be used interchangibly to mean different things, which can be confusing. So for all code written by me,
* **SPLIT**: A split of WLASL, one of asl100, asl300, asl1000 and asl2000
* **SET**: A subset of a given wlasl split, one of train, val test


# TODO:
- make a flag to control preprocessed vs unprocessed
- add removed instances
- add bounding box info

## Split

#### Pick split

In [2]:
split_options: List[stats.AVAIL_SPLITS] = ["asl100", "asl300", "asl1000", "asl2000"]
print('Options:')
for i, split_name in enumerate(split_options):
    print(f'{i} : {split_name}')

Options:
0 : asl100
1 : asl300
2 : asl1000
3 : asl2000


In [3]:
split_idx = 0 #change for different split
split_name: stats.AVAIL_SPLITS = split_options.pop(split_idx)
print(f'Selected: {split_name}')

Selected: asl100


doing this for SAICIST because those experiments kept bad frame range labeled images

In [4]:
# labels_dir = Path('./preprocessed/labels_new')
labels_dir = Path('./preprocessed/labels_old_nobbox') 
split_dir = labels_dir / split_name

with open(CLASSES_PATH, 'r') as f:
    classes = json.load(f)

## Preprocessed

Removed due to limited number of frames (cutoff 9):

Split :  Videos
asl1000 : 18223, 59958
asl2000 : 18223, 59958, 15144

if frame start or frame end were labeled wrong, they were set to 0 or frame length

In [5]:
set_info = stats.retrieve_split_data(split=split_name, labels_dir=labels_dir, pattern='*_instances*.json')
seperated = {}
set_keys = ['train', 'test', 'val']
print(f'Split info for {split_name}')
for  key, value in set_info.items():
    
    preproc_set = cast(List[preproc.InstanceDict], value)
    for set_key in set_keys:
        if key.startswith(set_key):
            seperated[set_key] = preproc_set
            print(f'{set_key}:')
            break
    
    print(f'Num instances: {len(preproc_set)}\n')

Split info for asl100
val:
Num instances: 338

train:
Num instances: 1442

test:
Num instances: 258



In [6]:
sep_wlasl_order = {}
for key, value in seperated.items():
    sep_wlasl_order[key] = stats.reverse_preproc_format(value, classes=classes)

In [7]:
per_set_stats = {}
for set_name, glosses_subset in sep_wlasl_order.items():
    per_set_stats[set_name] = stats.get_set_stats(glosses_subset)

In [8]:
per_set_stats = cast(Dict[stats.AVAIL_SETS, stats.set_stats], per_set_stats)

In [9]:
def get_min_max_num_instances(per_instance_stats: Dict[str, stats.instance_stats]) -> Tuple[int, int]:
    """Returns the minumum and maximum number of instances in a dictionary of instance stats

    Args:
        per_instance_stats (Dict[str, stats.instance_stats]): A dictionary with stats for each class indexed by label_name

    Returns:
        Tuple[int, int]: min, max number of instances.
    """
    mini = float('inf')
    maxi = float(0)
    for inst_stats in per_instance_stats.values():
        num_inst = inst_stats['num_instances']
        if num_inst > maxi:
            maxi = num_inst
        if num_inst < mini:
            mini = num_inst
    return int(mini), int(maxi) 



In [10]:



rows = []
for set_name, set_stats in per_set_stats.items():
    mini, maxi = get_min_max_num_instances(set_stats['per_instance_stats'])
    rows.append({
        'Set name': set_name,
        'Num instances': set_stats['num_instances'],
        'instances per class': f'[{mini} - {maxi}]'
    })

df = pd.DataFrame(rows)
df    # when this cell runs the dataframe is shown as a nicely aligned table


,Set name,Num instances,instances per class
0,val,338,[3 - 6]
1,train,1442,[12 - 30]
2,test,258,[2 - 5]


In [11]:
def create_instances_table(per_set_stats: Dict[stats.AVAIL_SETS, stats.set_stats]) -> pd.DataFrame:
    rows = []
    for set_name, set_stats in per_set_stats.items():
        mini, maxi = get_min_max_num_instances(set_stats['per_instance_stats'])
        rows.append({
            'Set name': set_name,
            'Num instances': set_stats['num_instances'],
            'instances per class': f'[{mini} - {maxi}]'
        })

    return pd.DataFrame(rows)
    

In [12]:
create_instances_table(per_set_stats)

,Set name,Num instances,instances per class
0,val,338,[3 - 6]
1,train,1442,[12 - 30]
2,test,258,[2 - 5]


In [14]:
for split_name in split_options:
    split_dir = labels_dir / split_name
    set_info = stats.retrieve_split_data(split=split_name, labels_dir=labels_dir, pattern='*_instances*.json')
    seperated = {}
    set_keys = ['train', 'test', 'val']
    print(f'Split info for {split_name}')
    for  key, value in set_info.items():
        preproc_set = cast(List[preproc.InstanceDict], value)
        for set_key in set_keys:
            if key.startswith(set_key):
                seperated[set_key] = preproc_set
                break
    
    sep_wlasl_order = {}
    for key, value in seperated.items():
        sep_wlasl_order[key] = stats.reverse_preproc_format(value, classes=classes)

    per_set_stats = {}
    for set_name, glosses_subset in sep_wlasl_order.items():
        per_set_stats[set_name] = stats.get_set_stats(glosses_subset)

    
    display(create_instances_table(per_set_stats))
    print('\n', '-'*10)

Split info for asl300


,Set name,Num instances,instances per class
0,val,901,[2 - 6]
1,train,3549,[9 - 30]
2,test,668,[2 - 5]



 ----------
Split info for asl1000


,Set name,Num instances,instances per class
0,val,2319,[1 - 6]
1,train,8977,[6 - 30]
2,test,1876,[1 - 5]



 ----------
Split info for asl2000


,Set name,Num instances,instances per class
0,val,3917,[1 - 6]
1,train,14290,[1 - 30]
2,test,2879,[1 - 5]



 ----------
